In [ ]:
import sys
import os
import numpy as np

# 1. Connect to Bluegrey Infrastructure
sys.path.append(os.path.abspath('..'))
from src.data.store import DataStore
from src.backtest.vector_backtester import PortfolioVectorEngine

# 2. Load Data
print("🗄️ Initializing Librarian...")
store = DataStore(library_name="fx.min")
target_asset = 'C:EURUSD'

# Pulling 3 months of 1-minute bars
df = store.load(target_asset, start_date="2023-10-01", end_date="2024-01-01")

if df is not None:
    print(f"✅ Loaded {len(df):,} rows of {target_asset}.")
    
    # ==========================================
    # 🧠 3. (EXAMPLE) THE LAB: Your Only Focus 
    # ==========================================
    # Parameters (in minutes)
    fast_window = 60   # 1 Hour
    slow_window = 240  # 4 Hours
    
    # Calculate Indicators
    df['sma_fast'] = df['close'].rolling(window=fast_window).mean()
    df['sma_slow'] = df['close'].rolling(window=slow_window).mean()
    
    # Initialize position target (0 = Flat)
    df['signal'] = 0.0 
    
    # The Rules
    df.loc[df['sma_fast'] > df['sma_slow'], 'signal'] = 1.0  # Golden Cross -> Long
    df.loc[df['sma_fast'] < df['sma_slow'], 'signal'] = -1.0 # Death Cross -> Short
    
    # Forward fill handles any NaNs during the initial rolling windows
    df['signal'] = df['signal'].replace(0.0, np.nan).ffill().fillna(0.0)
    
    # ==========================================
    # 📊 4. THE EVALUATION
    # ==========================================
    print("⚙️ Igniting Vector Engine...")
    
    # We pass the raw close prices and our target signals
    engine = PortfolioVectorEngine(
        prices=df['close'], 
        signals=df['signal'], 
        tc_bps=0.2,        # 0.2 bps spread assumption for EUR/USD
        execution_delay=1  # Wait 1 bar to execute to prevent look-ahead bias
    )
    
    engine.run()
    engine.tearsheet(title=f"SMA Crossover ({fast_window}m / {slow_window}m)")

else:
    print(f"❌ Could not load data for {target_asset}.")